<a href="https://colab.research.google.com/github/JSJeong-me/Retriever/blob/main/Phishing/Voice-Phishing01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install faiss-cpu transformers

In [2]:
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# JSON 파일 로드
json_file_path = "/content/law_enforcement_scam_0228.json"

with open(json_file_path, "r", encoding="utf-8") as f:
    data = json.load(f)


In [ ]:
# 텍스트 데이터 추출
texts = [entry["text"] for entry in data]
ids = [entry["id"] for entry in data]

# SentenceTransformer 모델 로드 (BERT 기반 임베딩)
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# 텍스트 데이터 임베딩 생성
embeddings = model.encode(texts, convert_to_numpy=True)

# FAISS Index 생성 (L2 거리 기반)
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)


In [4]:
# ID와 매핑
id_map = {i: ids[i] for i in range(len(ids))}

# 검색 함수
def search(query, top_k=5):
    query_vector = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_vector, top_k)

    results = []
    for i in range(top_k):
        idx = indices[0][i]
        results.append({"id": id_map[idx], "text": texts[idx], "distance": distances[0][i]})

    return results

In [5]:
# 예제 검색 실행
query = "계좌 개설 여부 확인"
results = search(query, top_k=3)

# 검색 결과 출력
for res in results:
    print(f"ID: {res['id']}, Text: {res['text']}, Distance: {res['distance']:.4f}")

ID: 00c2801d-17ea-4e55-9c6d-a5559b5c45ae, Text: 제가 간단한 본인 확인, Distance: 0.2297
ID: 76bfc089-ab59-4da8-90d2-e7031517864c, Text: 예 그러면 일단 1번 확인해 보시겠어요, Distance: 0.2659
ID: 76e14575-984f-4376-9189-9ee710094066, Text: 일단은 그렇게 수사관으로 팩트를 전달 받았는데요. 한번 확인해 볼게요., Distance: 0.2666


In [6]:
# 예제 검색 실행
query = "검찰"
results = search(query, top_k=3)

# 검색 결과 출력
for res in results:
    print(f"ID: {res['id']}, Text: {res['text']}, Distance: {res['distance']:.4f}")

ID: ccebe017-8cdd-47c1-9f7e-dcbb324d36d6, Text: 예. 김철, Distance: 0.3167
ID: 3e412a84-9047-4b6c-90b3-b1cdb5e2258c, Text: 그럼요, Distance: 0.3392
ID: acd82686-fc93-4937-84e3-05c0594c5673, Text: 그럼요., Distance: 0.3666


In [7]:
# 예제 검색 실행
query = "사기"
results = search(query, top_k=3)

# 검색 결과 출력
for res in results:
    print(f"ID: {res['id']}, Text: {res['text']}, Distance: {res['distance']:.4f}")

ID: 494d25d4-5e9f-4f13-af76-66e6546b283d, Text: 여기, Distance: 0.2802
ID: 7d19e314-b8be-40a3-8e3f-75dcc981db3a, Text: 일단, Distance: 0.3920
ID: 158dfe53-7a84-4260-8dba-aea29ea90428, Text: 제가 말씀드렸다시피 이런 소리가, Distance: 0.4343
